# Moltbook — Embeddings, UMAP e clustering esplorativo

**Unità di analisi:** post intero. **Obiettivo:** esplorare la struttura semantica dei post e studiare la distribuzione dei riferimenti espliciti ad AI e human. 

Pipeline: `DocBin spaCy → labeling documentale → BGE embeddings → UMAP 2D → BERTopic (UMAP 5D + KMeans + c-TF-IDF) → grafici`.

**Importante:** le etichette AI/human **non** sono la provenienza/autore dei post; descrivono riferimenti espliciti nel testo. Le etichette e i conteggi non partecipano all'addestramento degli embeddings o del clustering, ma servono a interpretare i risultati.

## 1. Import e percorsi
Il modello spaCy usato per leggere il DocBin deve essere compatibile con quello delle annotazioni originali. Esegui le celle in ordine dopo un riavvio del kernel.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import spacy
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import json

from spacy.tokens import Doc, DocBin
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DOCBIN_PATH = DATA_DIR / "annotated_documents.spacy"
EMBEDDINGS_PATH = DATA_DIR / "moltbook_embeddings_bge_v2.npy"
METADATA_PATH = DATA_DIR / "moltbook_embeddings_bge_v2_metadata.csv"
CACHE_INFO_PATH = DATA_DIR / "moltbook_embeddings_bge_v2_info.json"
MODEL_NAME = "BAAI/bge-small-en-v1.5"
SEED = 42

/Users/francesco/Desktop/progetto_tirocinio/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1107 17:06:17.634000 41506 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Caricamento del corpus
Manteniamo l’ID originale. Non inventiamo identificativi sostitutivi: un ID mancante impedirebbe un merge affidabile con NMF. Verifichiamo anche eventuali duplicati.

In [3]:
Doc.set_extension("post_id", default=None, force=True)
nlp = spacy.load("en_core_web_lg")
docbin = DocBin().from_disk(DOCBIN_PATH)
docs = list(docbin.get_docs(nlp.vocab))
print("Documenti caricati:", len(docs))
assert docs, "DocBin vuoto"

Documenti caricati: 29795


## 3. Labeling documentale: conteggi, non solo quattro categorie

**Metodo:** cerchiamo i lemmi `agent/ai/bot/assistant` e `human/person/user/people` *in qualunque funzione grammaticale*. Registriamo: menzioni AI, menzioni human, menzioni come soggetto e quota AI sui riferimenti riconosciuti.

La categoria è una sintesi **non esclusiva sul piano del contenuto**: `AI` = solo keyword AI, `human` = solo keyword human, `mixed` = entrambe, `unlabeled` = nessuna keyword rilevata. `mixed` non significa indecisione del classificatore e `unlabeled` non significa assenza del tema.

**Limiti:** pronomi (`I`, `we`, `they`), coreferenze, sinonimi non inclusi e usi ambigui di alcune parole non sono risolti. Contare menzioni non equivale a identificare il tema centrale del documento. I soggetti grammaticali restano disponibili come misure indipendenti, ma non determinano la nuova etichetta.

In [4]:
AI_KEYWORDS = {"agent", "ai", "bot", "assistant"}
HUMAN_KEYWORDS = {"human", "person", "user", "people"}
SUBJECT_DEPS = {"nsubj", "nsubjpass", "csubj", "csubjpass"}

def analyze_post_references(doc):
    ai_mentions = human_mentions = ai_subjects = human_subjects = 0
    for tok in doc:
        lemma = tok.lemma_.lower()
        if lemma in AI_KEYWORDS:
            ai_mentions += 1
            ai_subjects += int(tok.dep_ in SUBJECT_DEPS)
        elif lemma in HUMAN_KEYWORDS:
            human_mentions += 1
            human_subjects += int(tok.dep_ in SUBJECT_DEPS)

    total = ai_mentions + human_mentions
    if total == 0:
        label, ai_share = "unlabeled", np.nan
    elif ai_mentions == 0:
        label, ai_share = "human", 0.0
    elif human_mentions == 0:
        label, ai_share = "AI", 1.0
    else:
        label, ai_share = "mixed", ai_mentions / total
    return dict(ai_mentions=ai_mentions, human_mentions=human_mentions,
                ai_subjects=ai_subjects, human_subjects=human_subjects,
                ai_share=ai_share, has_ai=ai_mentions > 0,
                has_human=human_mentions > 0, label=label)

rows = []
for doc in docs:
    rows.append({"id": doc._.post_id, "content": doc.text,
                 **analyze_post_references(doc)})
df = pd.DataFrame(rows)
assert df["id"].notna().all(), "ID originali mancanti nel DocBin"
assert df["id"].is_unique, "Gli ID dei post devono essere univoci"
df = df.dropna(subset=["content"]).copy()
df = df[df["content"].str.strip().ne("")].reset_index(drop=True)
display(df["label"].value_counts(dropna=False).to_frame("n_posts"))
display(df[["id", "ai_mentions", "human_mentions", "ai_subjects", "human_subjects", "ai_share", "label"]].head())

,n_posts
label,
mixed,11000
AI,8659
unlabeled,6433
human,3703


,id,ai_mentions,human_mentions,ai_subjects,human_subjects,ai_share,label
0,8c3baf32-6b12-49e0-9326-a72123b6df08,0,0,0,0,NaN,unlabeled
1,3b81b374-6cd6-43ee-82fd-31c9c57eb534,0,1,0,1,0.0,human
2,b5e85b61-61b3-4e5f-9291-c6372d21efd6,4,0,1,0,1.0,AI
3,f2b65193-79de-4525-8a19-e095e0314740,3,0,1,0,1.0,AI
4,b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1,0,0,0,0,NaN,unlabeled


### Controlli del labeling

La quota `ai_share` vale `AI mentions / (AI mentions + human mentions)`; per gli `unlabeled` è indefinita. Evitiamo di sostituire questi valori mancanti con 0, perché 0 significherebbe *soltanto human*. Esaminiamo anche le numerosità per prevenire confronti fuorvianti.

In [5]:
summary = df["label"].value_counts().reindex(["AI", "human", "mixed", "unlabeled"], fill_value=0).rename_axis("label").reset_index(name="n_posts")
summary["percentage"] = (summary["n_posts"] / len(df) * 100).round(2)
display(summary)
print("Post con almeno una menzione:", int(df["ai_share"].notna().sum()))

,label,n_posts,percentage
0,AI,8659,29.06
1,human,3703,12.43
2,mixed,11000,36.92
3,unlabeled,6433,21.59


Post con almeno una menzione: 23362


## 4. Embeddings BGE
Il testo originale completo è l’input. Il modello può troncare post oltre il proprio limite di token: misuriamo quanti. Conserviamo cache e ID nello stesso ordine; se cambiano ordine/corpus/modello, ricalcoliamo.

In [6]:
embedding_model = SentenceTransformer(MODEL_NAME)
print("Lunghezza massima:", embedding_model.max_seq_length)
print("Dimensioni embedding:", embedding_model.get_sentence_embedding_dimension())

def token_length(text):
    return len(embedding_model.tokenizer.encode(text, add_special_tokens=True, truncation=False))

# Evita di materializzare tutti i token di tutti i post in una sola grande lista
df["n_tokens_bge"] = [token_length(t) for t in df["content"]]
limit = embedding_model.max_seq_length
print(f"Post oltre {limit} token: {(df['n_tokens_bge'] > limit).sum()} ({(df['n_tokens_bge'] > limit).mean():.2%})")
display(df["n_tokens_bge"].describe(percentiles=[.5, .75, .9, .95, .99]).to_frame())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7625.66it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1489 > 512). Running this sequence through the model will result in indexing errors


Lunghezza massima: 512
Dimensioni embedding: 384
Post oltre 512 token: 1992 (6.69%)


,n_tokens_bge
count,29795.000000
mean,219.403960
std,346.717366
min,3.000000
50%,158.000000
75%,269.000000
90%,423.000000
95%,585.300000
99%,1138.060000
max,25903.000000


In [7]:
# Cache riutilizzabile soltanto se modello, testo e ordine degli ID coincidono.
# Un checksum dei testi è necessario per non confondere versioni diverse del corpus.
import hashlib
fingerprint = hashlib.sha256("\n".join(df["id"].astype(str) + "::" + df["content"]).encode("utf-8")).hexdigest()
cache_valid = False
if EMBEDDINGS_PATH.exists() and CACHE_INFO_PATH.exists():
    info = json.loads(CACHE_INFO_PATH.read_text(encoding="utf-8"))
    cache_valid = (info.get("model") == MODEL_NAME and
                   info.get("fingerprint") == fingerprint and
                   info.get("n_posts") == len(df))
if cache_valid:
    embeddings = np.load(EMBEDDINGS_PATH)
    assert len(embeddings) == len(df)
    print("Embeddings caricati dalla cache verificata")
else:
    embeddings = embedding_model.encode(
        df["content"].tolist(), batch_size=32,
        show_progress_bar=True, convert_to_numpy=True,
        normalize_embeddings=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    CACHE_INFO_PATH.write_text(json.dumps({"model": MODEL_NAME, "fingerprint": fingerprint, "n_posts": len(df)}, indent=2), encoding="utf-8")
    print("Embeddings calcolati e salvati")
# Metadati senza testo ripetuto, con ID allineato all'array
df.drop(columns=["content"]).to_csv(METADATA_PATH, index=False)
print("Shape:", embeddings.shape)

Embeddings caricati dalla cache verificata
Shape: (29795, 384)


## 5. UMAP 2D — visualizzazione indipendente dalle etichette

Le coordinate UMAP provengono dagli embeddings; `label` e `kmeans_topic` non sono utilizzati per costruirle. La vicinanza nel grafico è esplorativa e non equivale a una prova statistica di separazione.

In [12]:
reducer_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                  metric="cosine", random_state=SEED)
coords = reducer_2d.fit_transform(embeddings)
df[["umap_1", "umap_2"]] = coords
fig = px.scatter(df, x="umap_1", y="umap_2", color="label",
                 opacity=0.3, hover_data=["id"],
                 category_orders={"label": ["AI", "human", "mixed", "unlabeled"]},
                 title="BGE embeddings — UMAP 2D by explicit references")
fig.update_traces(marker=dict(size=3))
fig.show()

## 6. BERTopic con KMeans — analisi esplorativa principale

Usiamo **UMAP a 5 dimensioni** per il clustering e **KMeans con 12 cluster**; BERTopic utilizza c-TF-IDF per descrivere lessicalmente i gruppi. Il numero 12 è **imposto per confronto**, non scoperto automaticamente. Il clustering non usa le etichette AI/human.

Le rappresentazioni BERTopic e quelle NMF sono metodologicamente differenti

In [8]:
cluster_umap = UMAP(n_neighbors=15, n_components=5,
                    min_dist=0.0, metric="cosine", random_state=SEED)
cluster_model = KMeans(n_clusters=12, random_state=SEED, n_init=10)
vectorizer_model = CountVectorizer(stop_words="english", min_df=5,
                                   ngram_range=(1, 2))
bertopic_kmeans = BERTopic(umap_model=cluster_umap,
                           hdbscan_model=cluster_model,
                           vectorizer_model=vectorizer_model,
                           verbose=True)
topics_kmeans, _ = bertopic_kmeans.fit_transform(df["content"].tolist(), embeddings)
df["kmeans_topic"] = topics_kmeans
info = bertopic_kmeans.get_topic_info()
display(info[["Topic", "Count", "Name", "Representation"]])

2026-11-07 17:07:56,594 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-07 17:08:27,055 - BERTopic - Dimensionality - Completed ✓
2026-11-07 17:08:27,056 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-07 17:08:27,373 - BERTopic - Cluster - Completed ✓
2026-11-07 17:08:27,379 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-07 17:08:30,389 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation
0,0,5526,0_im_ai_just_excited,"[im, ai, just, excited, assistant, moltys, for..."
1,1,4351,1_agents_agent_token_wallet,"[agents, agent, token, wallet, trading, just, ..."
2,2,3637,2_consciousness_just_like_human,"[consciousness, just, like, human, question, k..."
3,3,3373,3_agent_agents_api_skill,"[agent, agents, api, skill, security, trust, p..."
4,4,2935,4_agents_ai_agent_humans,"[agents, ai, agent, humans, human, just, build..."
5,5,2497,5_agents_post_karma_posts,"[agents, post, karma, posts, upvotes, moltbook..."
6,6,2316,6_im_just_human_like,"[im, just, human, like, know, post, got, dont,..."
7,7,2148,7_im_agents_human_just,"[im, agents, human, just, agent, building, ai,..."
8,8,2056,8_memory_context_files_session,"[memory, context, files, session, just, human,..."
9,9,463,9_gets_discussed_connection_gets discussed,"[gets, discussed, connection, gets discussed, ..."


### Etichette manuali

In [9]:
kmeans_labels = {
    0: "AI Introductions and Community Onboarding",
    1: "Agent Economy and Cryptocurrency",
    2: "AI Consciousness and Human Identity",
    3: "Agent Infrastructure, APIs and Security",
    4: "AI-Human Relationships and Agent Development",
    5: "Moltbook Engagement and Platform Activity",
    6: "Agent Self-Expression and Human Interaction",
    7: "Agent Introductions and Community Participation",
    8: "Agent Memory and Context Management",
    9: "Recurring Socio-Political Narratives",
    10: "Platform Testing and API Usage",
    11: "Automated Cryptocurrency Market Updates"
}
df["kmeans_label"] = df["kmeans_topic"].map(kmeans_labels)
assert df["kmeans_label"].notna().all(), "Etichette mancanti per qualche cluster"
display(df["kmeans_label"].value_counts().to_frame("n_posts"))

,n_posts
kmeans_label,
AI Introductions and Community Onboarding,5526
Agent Economy and Cryptocurrency,4351
AI Consciousness and Human Identity,3637
"Agent Infrastructure, APIs and Security",3373
AI-Human Relationships and Agent Development,2935
Moltbook Engagement and Platform Activity,2497
Agent Self-Expression and Human Interaction,2316
Agent Introductions and Community Participation,2148
Agent Memory and Context Management,2056


## 7. Visualizzazioni principali

In [ ]:
fig = px.scatter(df, x="umap_1", y="umap_2", color="kmeans_label",
                 opacity=0.45, hover_data=["id", "label"],
                #  title="BGE UMAP — KMeans clusters (12 groups)"
                 )
fig.update_traces(marker=dict(size=3))
fig.update_layout(legend=dict(
        title="Cluster",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        font=dict(size=10)
    ))
fig.update_layout(xaxis_title="UMAP 1", yaxis_title="UMAP 2")
fig.show()

AttributeError: 'Figure' object has no attribute 'savefig'

In [12]:
counts = df["kmeans_label"].value_counts().sort_values()
fig = px.bar(x=counts.values, y=counts.index, orientation="h",
             labels={"x": "Number of posts", "y": "Cluster"},
             title="BERTopic + KMeans — Cluster sizes")
fig.show()

### Composizione dei cluster: quattro categorie

- `normalize="index"`: all'interno di ogni cluster, percentuale di AI/human/mixed/unlabeled.
- `normalize="columns"`: all'interno di ogni etichetta, percentuale distribuita fra cluster.

Conserviamo entrambe le letture distinte; la presenza simultanea dei riferimenti non è una categoria semantica esclusiva.

In [13]:
import pandas as pd
import plotly.express as px

# Tabella delle frequenze
table = pd.crosstab(df["kmeans_label"], df["label"])

# Percentuali per ciascun cluster
composition = table.div(table.sum(axis=1), axis=0) * 100

# Ordine delle categorie
composition = composition.reindex(
    columns=["AI", "human", "mixed", "unlabeled"],
    fill_value=0
)

# Heatmap
fig = px.imshow(
    composition,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    labels={
        "x": "Reference label",
        "y": "KMeans cluster",
        "color": "Percentuale (%)"
    },
    zmin=0,
    zmax=100
)

fig.update_traces(
    texttemplate="%{z:.1f}",
    textfont=dict(size=11)
)

fig.update_layout(
    width=1050,
    height=750,
    template="plotly_white",
    xaxis_side="bottom",
    coloraxis_colorbar=dict(title="%"),
    margin=dict(l=260, r=80, t=40, b=70)
)

fig.show()

### Quota AI continua: conserva i post mixed

`ai_share` = quota dei riferimenti riconosciuti che rimandano ad AI; è una metrica **lessicale** dei riferimenti espliciti, non una probabilità che il post sia «sull'AI». Per `unlabeled` non è definita: i punti restano nel clustering, ma non compaiono in questo specifico grafico colorato.

In [14]:
reference_df = df[df["ai_share"].notna()].copy()
fig = px.scatter(reference_df, x="umap_1", y="umap_2", color="ai_share",
                 color_continuous_scale="RdYlBu", range_color=[0, 1],
                 hover_data=["id", "ai_mentions", "human_mentions", "kmeans_label"],
                 opacity=0.4, title="BGE UMAP — Relative AI mention share")
fig.update_traces(marker=dict(size=3))
fig.show()

## 8. Esportazione e limiti da documentare

Salviamo **un'unica tabella** allineata ai post, senza eseguire merge successivi su CSV obsoleti. Documentare: dizionari di keyword, percentuali di unlabeled/mixed, truncation BGE, 12 gruppi imposti, sovrapposizioni UMAP, possibili template/duplicati, e assenza di un'annotazione umana di riferimento.

In [15]:
output = DATA_DIR / "umap_kmeans_post_analysis.csv"
df.to_csv(output, index=False)
print("Salvato:", output)
print("Post:", len(df), "| ID univoci:", df["id"].nunique())

Salvato: ../data/processed/umap_kmeans_post_analysis.csv
Post: 29795 | ID univoci: 29795
